# SQL Practice #17: Calculate Total Billing per Employee

**Question:** Given a `billings` table (with bill rates effective from a `bill_date`) and an `hoursworked` table (with hours logged on specific dates), calculate the **total billing amount per employee**. Each work record should use the bill rate that was in effect on that work date.

**Key Insight:** Use `LEAD()` to find the next bill date per employee, then join `hoursworked` to `billings` where the `work_date` falls between `bill_date` and `next_bill_date - 1`.

**Approach:**
1. Create the `billings` and `hoursworked` tables with sample data (Cell 1–3)
2. Solve with SQL using `LEAD()` window function and a date-range join (Cell 4)
3. Reproduce the same solution in PySpark and aggregate totals (Cell 5–7)

In [0]:
%sql
-- ============================================================
-- Cell 1: Create billings and hoursworked tables with sample data
-- ============================================================
DROP TABLE IF EXISTS b_sql.b_practice.billings;

CREATE TABLE b_sql.b_practice.billings (
    emp_name VARCHAR(10),
    bill_date DATE,
    bill_rate INT
);

INSERT INTO b_sql.b_practice.billings VALUES
('Sachin', '1990-01-01', 25),
('Sehwag', '1989-01-01', 15),
('Dhoni',  '1989-01-01', 20),
('Sachin', '1991-02-05', 30);


DROP TABLE IF EXISTS b_sql.b_practice.hoursworked;

CREATE TABLE b_sql.b_practice.hoursworked (
    emp_name VARCHAR(20),
    work_date DATE,
    bill_hrs INT
);

INSERT INTO b_sql.b_practice.hoursworked VALUES
('Sachin',  '1990-07-01', 3),
('Sachin',  '1990-08-01', 5),
('Sehwag',  '1990-07-01', 2),
('Sachin',  '1991-07-01', 4);


In [0]:
%sql
-- ============================================================
-- Cell 2: Preview the billings table
-- ============================================================
select * from b_sql.b_practice.billings

In [0]:
%sql
-- ============================================================
-- Cell 3: Preview the hoursworked table
-- ============================================================
select * from b_sql.b_practice.hoursworked

In [0]:
%sql
-- ============================================================
-- Cell 4: SQL Solution
-- Calculate billing amount per work record using the applicable bill rate.
-- LEAD() gets the next bill_date; work_date must fall between
-- bill_date and next_bill_date (next bill_date - 1 day).
-- ============================================================
with bill_date as (SELECT *,
       LEAD(
           date_add(bill_date, -1),
           1,
           DATE('9999-12-31')
       ) OVER (
           PARTITION BY emp_name
           ORDER BY bill_date
       ) AS next_bill_date
FROM b_sql.b_practice.billings order by emp_name, bill_date)
-- Join hoursworked to billing periods; calculate rate = bill_rate * bill_hrs
select b.*, h.bill_hrs,(b.bill_rate*h.bill_hrs) as rate  from b_sql.b_practice.hoursworked h join bill_date b on h.emp_name=b.emp_name and h.work_date between b.bill_date and b.next_bill_date 


In [0]:
# ============================================================
# Cell 5: Load billings and hoursworked into PySpark DataFrames
# ============================================================
df_billing= spark.read.table("b_sql.b_practice.billings")
df_households= spark.read.table("b_sql.b_practice.hoursworked")
df_billing.display()
df_households.display()


In [0]:
# ============================================================
# Cell 6: PySpark Solution
# Calculate billing amount per work record using the applicable bill rate.
# lead() gets the next bill_date; work_date must fall between
# bill_date and next_bill_date (next bill_date - 1 day).
# ============================================================
from pyspark.sql.window import Window
from pyspark.sql.functions import *

# Window spec: order billing records by date within each employee
bill_window = (
    Window
    .partitionBy("emp_name")
    .orderBy("bill_date")
)

# Add next_bill_date: the day before the next billing record (or 9999-12-31 if last)
df_bill_date = (
    df_billing
    .withColumn(
        "next_bill_date",
        lead(
            date_add(col("bill_date"), -1),
            1,
            lit("9999-12-31").cast("date")
        ).over(bill_window)
    )
)

# Join hoursworked to billings where work_date falls within the billing period
df_result = (
    df_households.alias("h")
    .join(
        df_bill_date.alias("b"),
        (col("h.emp_name") == col("b.emp_name")) &
        col("h.work_date").between(
            col("b.bill_date"),
            col("b.next_bill_date")
        ),
        "inner"
    )
    # Select work details and compute rate = hours * bill_rate
    .select(
        col("h.emp_name"),
        col("h.work_date"),
        col("h.bill_hrs"),
        col("b.bill_rate"),
        (col("h.bill_hrs") * col("b.bill_rate")).alias("rate")
    )
)

df_result.display()


In [0]:
# ============================================================
# Cell 7: Aggregate total billing amount per employee
# ============================================================
df_final=df_result.groupBy("emp_name").agg(sum("rate").alias("total"))
df_final.display()